In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
device.type

'cpu'

In [3]:
%load_ext autoreload
%autoreload 2

from drGAT import drGAT

In [4]:
train_data = pd.read_csv("../CTRP_data/train.csv")
val_data = pd.read_csv("../CTRP_data/val.csv")
test_data = pd.read_csv("../CTRP_data/test.csv")

In [5]:
train_data.head()

,Drug,Cell
0,YM-155,SKM-1
1,austocystin D,RCC10RGB
2,sorafenib,SK-HEP-1
3,UNC0638,RT4
4,cabozantinib,SNU-5


In [6]:
idxs = np.load("../CTRP_data/idxs.npy", allow_pickle=True)
converter = {idxs[1, i]: int(idxs[0, i]) for i in range(idxs.shape[1])}

In [7]:
edge_index = np.load("../CTRP_data/edge_idxs.npy")
edge_attr = np.load("../CTRP_data/edge_attr.npy")

In [8]:
def get_idx(X):
    X["Drug"] = [converter[(i)] for i in X["Drug"]]
    X["Cell"] = [converter[(i)] for i in X["Cell"]]
    return X

In [9]:
train_data = get_idx(train_data)
val_data = get_idx(val_data)
test_data = get_idx(test_data)

In [10]:
edge_index = torch.tensor(edge_index).int()
edge_index = edge_index.type(torch.int64)

In [11]:
edge_attr = torch.tensor(edge_attr).float()

In [12]:
train_drug = train_data["Drug"].values
train_cell = train_data["Cell"].values
val_drug = val_data["Drug"].values
val_cell = val_data["Cell"].values

In [13]:
train_labels = np.load("../CTRP_data/train_labels.npy")
val_labels = np.load("../CTRP_data/val_labels.npy")

train_labels = torch.tensor(train_labels).float()
val_labels = torch.tensor(val_labels).float()

## Get feature matrix

In [14]:
drug = pd.read_csv("../CTRP_data/drug_sim.csv", index_col=0)
cell = pd.read_csv("../CTRP_data/cell_sim.csv", index_col=0)
gene = pd.read_csv("../CTRP_data/gene_sim.csv", index_col=0)

In [15]:
drug = torch.tensor(drug.values).float()
cell = torch.tensor(cell.values).float()
gene = torch.tensor(gene.values).float()

# Create the dataset

In [16]:
data = [
    drug,
    cell,
    gene,
    edge_index,
    edge_attr,
    train_drug,
    train_cell,
    val_drug,
    val_cell,
    train_labels,
    val_labels,
]

In [17]:
params = {
    "dropout1": 0.1,
    "dropout2": 0.1,
    "n_drug": drug.shape[0],
    "n_cell": cell.shape[0],
    "n_gene": gene.shape[0],
    "hidden1": 256,
    "hidden2": 32,
    "hidden3": 128,
    "epochs": 3,
    "lr": 0.001,
    "heads": 2,
}

In [18]:
def objective(trial):
    params = {
        "n_drug": drug.shape[0],
        "n_cell": cell.shape[0],
        "n_gene": gene.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_int("epochs", 10, 200, step=50),
        "heads": trial.suggest_categorical("heads", [1, 2, 3]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer", ["GAT", "GATv2", "Transformer"]
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        _, _, _, best_metrics, early_stopping_epoch = drGAT.train(
            data, params=params, device=device, verbose=False
        )
        print("#####")
        print(best_metrics)
        print("#####")

        early_stop_threshold = trial.suggest_float("early_stop_threshold", 0.3, 0.7)
        if (
            early_stopping_epoch is not None
            and early_stopping_epoch < params["epochs"] * early_stop_threshold
        ):
            raise optuna.TrialPruned("Early stopping occurred too early")

        trial.set_user_attr("early_stopping_epoch", early_stopping_epoch)
        return best_metrics

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print("CUDA out of memory")
            trial.set_user_attr("status", "CUDA OOM")

            torch.cuda.empty_cache()
            gc.collect()

            return [None] * 4
        else:
            raise e

In [19]:
name = "CTRP_GAT"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-21 13:47:53,646] A new study created in RDB with name: CTRP_GAT


Using:  cpu


/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:11<00:00,  5.56s/it]
[I 2025-03-21 13:48:05,133] Trial 0 finished with values: [0.6555796577488552, 0.8492063314278666, 0.8762479449380195, 0.7384699853587116] and parameters: {'dropout1': 0.2, 'dropout2': 0.5, 'heads': 3, 'activation': 'swish', 'optimizer': 'Adam', 'lr': 0.0001211315623580244, 'weight_decay': 0.00010953603288263856, 'scheduler': 'Step', 'gnn_layer': 'Transformer', 'gamma_step': 0.20636410588222365, 'step_size': 29, 'amsgrad': False, 'early_stop_threshold': 0.3105551177177229}.
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn

Best model found at epoch 2
#####
[0.6555796577488552, 0.8492063314278666, 0.8762479449380195, 0.7384699853587116]
#####
Using:  cpu


100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:12<00:00,  6.31s/it]
[I 2025-03-21 13:48:17,827] Trial 1 finished with values: [0.3078452639190166, 0.39133919673239137, 0.2786528474368343, 0.3259198403849539] and parameters: {'dropout1': 0.3, 'dropout2': 0.5, 'heads': 3, 'activation': 'swish', 'optimizer': 'SGD', 'lr': 5.043696840237797e-05, 'weight_decay': 0.0047643642658817195, 'scheduler': 'Step', 'gnn_layer': 'GAT', 'gamma_step': 0.7282148840061975, 'step_size': 14, 'momentum': 0.8229194700403771, 'nesterov': True, 'early_stop_threshold': 0.6810843536098026}.
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


...

Best model found at epoch 2
#####
[0.5, 0.8718750241790344, 0.8872618964423619, 0.0]
#####
Using:  cpu


100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:12<00:00,  6.36s/it]
[I 2025-03-21 14:08:40,849] Trial 98 finished with values: [0.5003012774162449, 0.8664532825370124, 0.8807546673237985, 0.0016853256289876008] and parameters: {'dropout1': 0.2, 'dropout2': 0.3, 'heads': 3, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.0029513541118979833, 'weight_decay': 0.0024977349457427998, 'scheduler': 'Cosine', 'gnn_layer': 'Transformer', 'T_max': 26, 'amsgrad': True, 'early_stop_threshold': 0.40176658379863295}.
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Best model found at epoch 2
#####
[0.5003012774162449, 0.8664532825370124, 0.8807546673237985, 0.0016853256289876008]
#####
Using:  cpu


100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:11<00:00,  5.50s/it]
[I 2025-03-21 14:08:51,954] Trial 99 finished with values: [0.5, 0.583929560543889, 0.6079219055824897, 0.6666666666666666] and parameters: {'dropout1': 0.3, 'dropout2': 0.4, 'heads': 2, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.001865273587015343, 'weight_decay': 0.0009967853336365156, 'scheduler': 'Cosine', 'gnn_layer': 'GATv2', 'T_max': 32, 'amsgrad': True, 'early_stop_threshold': 0.4667116813814679}.


Best model found at epoch 2
#####
[0.5, 0.583929560543889, 0.6079219055824897, 0.6666666666666666]
#####


## Eval

In [21]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [22]:
# prob, res, test_attention = drGAT.eval(model, test)